# SOOP Data Diagnostic

Check SOOP data quality: shapes, values, mask alignment.

**Setup:** Add SOOP notebook output as Input.

In [ ]:
import os, sys
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

# Find and unpack SOOP
tar_path = None
for item in Path("/kaggle/input").iterdir():
    for f in item.rglob("soop_ds004889.tar"):
        tar_path = f
        break
    if tar_path: break

SOOP_ROOT = Path("/tmp/soop/ds004889")
if not SOOP_ROOT.exists() and tar_path:
    os.makedirs("/tmp/soop", exist_ok=True)
    print(f"Unpacking {tar_path}...")
    !tar xf {tar_path} -C /tmp/soop/
    print("Done!")

soop_subs = sorted([d.name for d in SOOP_ROOT.iterdir() if d.name.startswith("sub-")])
print(f"SOOP subjects: {len(soop_subs)}")

In [ ]:
# Check file structure for first 5 subjects
for sub_id in soop_subs[:5]:
    sub_dir = SOOP_ROOT / sub_id
    dwi_dir = sub_dir / "dwi"
    anat_dir = sub_dir / "anat"
    mask_dir = SOOP_ROOT / "derivatives" / "lesion_masks" / sub_id / "dwi"
    
    print(f"\n=== {sub_id} ===")
    if dwi_dir.exists():
        for f in sorted(dwi_dir.iterdir()):
            print(f"  dwi/{f.name} ({f.stat().st_size/1e6:.1f} MB)")
    if anat_dir.exists():
        for f in sorted(anat_dir.iterdir()):
            print(f"  anat/{f.name} ({f.stat().st_size/1e6:.1f} MB)")
    if mask_dir.exists():
        for f in sorted(mask_dir.iterdir()):
            print(f"  mask/{f.name} ({f.stat().st_size/1e6:.1f} MB)")
    else:
        print("  NO MASK!")

In [ ]:
def load_soop_subject(sub_id):
    """Load raw NIfTI data for a SOOP subject."""
    sub_dir = SOOP_ROOT / sub_id
    mask_dir = SOOP_ROOT / "derivatives" / "lesion_masks" / sub_id / "dwi"
    
    dwi_path = sub_dir / "dwi" / f"{sub_id}_rec-TRACE_dwi.nii.gz"
    adc_path = sub_dir / "dwi" / f"{sub_id}_rec-ADC_dwi.nii.gz"
    flair_path = sub_dir / "anat" / f"{sub_id}_FLAIR.nii.gz"
    
    # Find mask
    mask_acute = mask_dir / f"{sub_id}_space-TRACE_desc-lesionAcute_mask.nii.gz"
    mask_combined = mask_dir / f"{sub_id}_space-TRACE_desc-lesion_mask.nii.gz"
    mask_path = mask_acute if mask_acute.exists() else mask_combined if mask_combined.exists() else None
    
    result = {}
    for name, path in [("dwi", dwi_path), ("adc", adc_path), ("flair", flair_path)]:
        if path.exists():
            img = nib.load(path)
            data = img.get_fdata(dtype=np.float32)
            result[name] = {"data": data, "img": img, "path": path}
            print(f"  {name}: shape={data.shape}, dtype={data.dtype}, "
                  f"range=[{data.min():.1f}, {data.max():.1f}], "
                  f"spacing={img.header.get_zooms()[:3]}")
        else:
            print(f"  {name}: MISSING ({path})")
    
    if mask_path and mask_path.exists():
        img = nib.load(mask_path)
        data = img.get_fdata(dtype=np.float32)
        result["mask"] = {"data": data, "img": img, "path": mask_path}
        n_voxels = (data > 0).sum()
        print(f"  mask: shape={data.shape}, lesion_voxels={n_voxels}, "
              f"spacing={img.header.get_zooms()[:3]}, "
              f"file={mask_path.name}")
    else:
        print(f"  mask: MISSING")
    
    return result

In [ ]:
# Load first 5 subjects with masks
subjects_with_masks = []
for sub_id in soop_subs:
    mask_dir = SOOP_ROOT / "derivatives" / "lesion_masks" / sub_id / "dwi"
    if mask_dir.exists() and any(mask_dir.iterdir()):
        subjects_with_masks.append(sub_id)
    if len(subjects_with_masks) >= 10:
        break

print(f"First 10 subjects with masks: {subjects_with_masks}")

# Load 3 subjects
loaded = {}
for sub_id in subjects_with_masks[:3]:
    print(f"\n=== Loading {sub_id} ===")
    try:
        loaded[sub_id] = load_soop_subject(sub_id)
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
# KEY CHECK: Do DWI and mask have the same shape?
print("Shape compatibility check:")
print("="*60)
for sub_id, data in loaded.items():
    if "dwi" in data and "mask" in data:
        dwi_shape = data["dwi"]["data"].shape
        mask_shape = data["mask"]["data"].shape
        match = "OK" if dwi_shape == mask_shape else "MISMATCH!"
        print(f"{sub_id}: DWI={dwi_shape}, mask={mask_shape} -> {match}")
        
        if "adc" in data:
            adc_shape = data["adc"]["data"].shape
            print(f"  ADC={adc_shape} {'OK' if adc_shape == dwi_shape else 'MISMATCH!'}")
        if "flair" in data:
            flair_shape = data["flair"]["data"].shape
            print(f"  FLAIR={flair_shape} {'OK' if flair_shape == dwi_shape else 'DIFFERENT (expected)'}")

In [ ]:
# Visualize DWI + mask overlay for each loaded subject
for sub_id, data in loaded.items():
    if "dwi" not in data or "mask" not in data:
        continue
    
    dwi = data["dwi"]["data"]
    mask = data["mask"]["data"]
    
    # Handle 4D
    if dwi.ndim == 4:
        dwi = dwi[..., 0]
    if mask.ndim == 4:
        mask = mask[..., 0]
    
    # If shapes differ, resize mask to DWI
    from scipy.ndimage import zoom
    if mask.shape != dwi.shape:
        factors = [d/m for d, m in zip(dwi.shape, mask.shape)]
        mask = zoom(mask, factors, order=0)
    
    mask_bin = (mask > 0).astype(float)
    n_lesion = int(mask_bin.sum())
    
    # Find slice with most lesion voxels
    lesion_per_slice = mask_bin.sum(axis=(0, 1))  # sum over x,y for each z
    if lesion_per_slice.max() > 0:
        best_z = int(np.argmax(lesion_per_slice))
    else:
        best_z = dwi.shape[2] // 2
    
    # Show 5 slices around the best
    n_slices = 5
    slices = [max(0, min(dwi.shape[2]-1, best_z + i - 2)) for i in range(n_slices)]
    
    fig, axes = plt.subplots(2, n_slices, figsize=(18, 7))
    fig.suptitle(f"{sub_id} | lesion={n_lesion} voxels | DWI shape={dwi.shape}", fontsize=14)
    
    for i, z in enumerate(slices):
        # DWI only
        axes[0, i].imshow(dwi[:, :, z].T, cmap="gray", origin="lower")
        axes[0, i].set_title(f"DWI z={z}")
        axes[0, i].axis("off")
        
        # DWI + mask overlay
        axes[1, i].imshow(dwi[:, :, z].T, cmap="gray", origin="lower")
        if mask_bin[:, :, z].sum() > 0:
            axes[1, i].imshow(mask_bin[:, :, z].T, cmap="Reds", alpha=0.4, origin="lower")
        axes[1, i].set_title(f"DWI+mask z={z}")
        axes[1, i].axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Also check: ADC and FLAIR for same subject
for sub_id, data in loaded.items():
    if not all(k in data for k in ["dwi", "adc", "flair"]):
        continue
    
    dwi = data["dwi"]["data"]
    adc = data["adc"]["data"]
    flair = data["flair"]["data"]
    if dwi.ndim == 4: dwi = dwi[..., 0]
    if adc.ndim == 4: adc = adc[..., 0]
    if flair.ndim == 4: flair = flair[..., 0]
    
    z_dwi = dwi.shape[2] // 2
    z_flair = flair.shape[2] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"{sub_id} | DWI={dwi.shape} ADC={adc.shape} FLAIR={flair.shape}")
    
    axes[0].imshow(dwi[:, :, z_dwi].T, cmap="gray", origin="lower")
    axes[0].set_title(f"DWI z={z_dwi}")
    axes[0].axis("off")
    
    axes[1].imshow(adc[:, :, z_dwi].T, cmap="gray", origin="lower")
    axes[1].set_title(f"ADC z={z_dwi}")
    axes[1].axis("off")
    
    axes[2].imshow(flair[:, :, z_flair].T, cmap="gray", origin="lower")
    axes[2].set_title(f"FLAIR z={z_flair}")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    break  # just first subject

In [ ]:
# Statistics across more subjects: how many have masks with actual lesions?
mask_stats = []
for sub_id in subjects_with_masks:
    mask_dir = SOOP_ROOT / "derivatives" / "lesion_masks" / sub_id / "dwi"
    mask_acute = mask_dir / f"{sub_id}_space-TRACE_desc-lesionAcute_mask.nii.gz"
    mask_combined = mask_dir / f"{sub_id}_space-TRACE_desc-lesion_mask.nii.gz"
    mask_path = mask_acute if mask_acute.exists() else mask_combined
    
    try:
        img = nib.load(mask_path)
        data = img.get_fdata(dtype=np.float32)
        n_voxels = int((data > 0).sum())
        mask_stats.append({"sub": sub_id, "voxels": n_voxels, "shape": data.shape})
    except Exception as e:
        mask_stats.append({"sub": sub_id, "voxels": -1, "error": str(e)})

print(f"\nMask statistics for {len(mask_stats)} subjects:")
for s in mask_stats:
    status = f"{s['voxels']} voxels" if s['voxels'] >= 0 else f"ERROR: {s.get('error', '?')}"
    print(f"  {s['sub']}: {status}")

valid = [s for s in mask_stats if s['voxels'] > 0]
empty = [s for s in mask_stats if s['voxels'] == 0]
errors = [s for s in mask_stats if s['voxels'] < 0]
print(f"\nWith lesion: {len(valid)}, Empty mask: {len(empty)}, Errors: {len(errors)}")